# Interactive YNet lipid-mask erosion comparison

Compare one YNet on the same measured CSI data using freshly computed lipid operators. Positive `n` grows the lipid mask; negative `n` grows the brain mask and shrinks the lipid mask. Nothing is written to disk.

In [ ]:
from pathlib import Path

import sys

PROJECT_ROOT = Path('..').resolve()
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

import matplotlib.pyplot as plt
import numpy as np
import torch

from walinet.inference.inference import runNNLipRemoval2
from walinet.paper_figures.mask_degradation.pipeline import _compute_operator
from walinet.visualization.EvaluateModel import (
    interactive_spectra_viewer_multi,
    make_ppm_axis,
)

# ---- Settings -------------------------------------------------------------
scanner = "7T"
subject = "B0corrected_wo_LipidMask/Vol1_UCSF"
model_name = "YNet_ablation"
checkpoint = "model_last.pt"
n_values = (0, -4)
gpu_index = 0
batch_size = 200

repo_root = Path.cwd().resolve()
if repo_root.name == "notebooks":
    repo_root = repo_root.parent
data_root = repo_root / "data"
model_root = repo_root / "models"
subject_dir = data_root / scanner / subject
device = torch.device(f"cuda:{gpu_index}" if torch.cuda.is_available() else "cpu")
print("Subject:", subject_dir)
print("Model:", model_root / model_name)
print("Device:", device)

In [ ]:
fid = np.load(subject_dir / "OriginalData/data.npy")
magnitude = np.load(subject_dir / "OriginalData/magnitude.npy")
brain_mask = np.load(subject_dir / "masks/brain_mask.npy") > 0
lipid_mask = np.load(subject_dir / "masks/lipid_mask.npy") > 0
head_mask = brain_mask | lipid_mask
spectra = np.fft.fftshift(np.fft.fft(fid, axis=-1), axes=-1).astype(np.complex64)
del fid
n_timepoints = spectra.shape[-1]

operators = {}
cleaned = {}
mask_remaining_percent = {}

for n in n_values:
    print(f"\n=== n={n} ===")
    operator, original_count, changed_count, transferred_count = _compute_operator(
        subject_dir=subject_dir,
        input_data="OriginalData/data.npy",
        brain_mask_path="masks/brain_mask.npy",
        lipid_mask_path="masks/lipid_mask.npy",
        n_timepoints=n_timepoints,
        n_voxels=n,
        device=device,
        target=0.997,
    )
    remaining = 100.0 * min(changed_count, original_count) / original_count
    mask_remaining_percent[n] = remaining
    operators[n] = operator
    print(
        f"Lipid mask remaining: {remaining:.2f}% "
        f"({changed_count}/{original_count} voxels; transferred: {transferred_count})"
    )
    cleaned[n] = runNNLipRemoval2(
        device=device,
        exp=model_name,
        spectra=spectra,
        LipidProj_Operator_ff=operator,
        headmask=head_mask,
        batch_size=batch_size,
        normalization="auto",
        architecture="ynet",
        model_root=model_root,
        checkpoint=checkpoint,
    ).astype(np.complex64, copy=False)

print("\nReady:", {n: f"{mask_remaining_percent[n]:.2f}%" for n in n_values})

In [ ]:
%matplotlib widget

ppm = make_ppm_axis(
    vec_size=n_timepoints,
    dwelltime=360000,
    larmor_freq=297222876,
    ppm_ref=4.7,
    ref_freq_hz=0.0,
    dwelltime_unit="ns",
)

def spectrum_func(volume):
    return lambda x, y, z: np.real(volume[x, y, z, :])

def nuisance_func(cleaned_volume):
    return lambda x, y, z: np.real(
        spectra[x, y, z, :] - cleaned_volume[x, y, z, :]
    )

reference_n = n_values[0]
other_n = n_values[1:]
reference_label = f"n={reference_n} ({mask_remaining_percent[reference_n]:.1f}% lipid)"
other_labels = [
    f"n={n} ({mask_remaining_percent[n]:.1f}% lipid)" for n in other_n
]

interactive_spectra_viewer_multi(
    image_volume=magnitude,
    spectrum_funcs=[
        spectrum_func(spectra),
        nuisance_func(cleaned[reference_n]),
        spectrum_func(cleaned[reference_n]),
    ],
    labels=["Input and predicted nuisance", "Clean spectra"],
    overlay_labels=["Input", reference_label],
    extra_spectrum_funcs=[
        [nuisance_func(cleaned[n]) for n in other_n],
        [spectrum_func(cleaned[n]) for n in other_n],
    ],
    extra_labels=[other_labels, other_labels],
    x_axis=ppm,
    x_label="ppm",
    ppm_min=0,
    ppm_max=4,
    compact=True,
    metabolite_lines={
        "water": 4.7, "Cr": 3.02, "GABA": 2.28, "GPC": 3.23,
        "GSH": 2.95, "Gln": 2.45, "Glu": 2.35, "Ins": 3.56,
        "NAA": 2.01, "NAAG": 2.04, "PCh": 3.22, "PCr": 3.93,
        "Scyllo": 3.35, "Tau": 3.42, "2HG": 2.25, "Gly": 3.55,
    },
)